# Phase 3: Data Understanding & Initial Quality Assessment
## IBM Data Analyst Professional Certificate Capstone Project

**Author**: Raghavendra Golla  
**Role**: Associate Data Analyst  
**Project Theme**: Technology Trends & Skills Analysis for Global IT Consulting  

---

### 1. Executive Context & Business Objective
Before performing data cleaning, exploratory data analysis (EDA), or building dashboards, a professional Data Analyst must perform **Exploratory Data Assessment / Data Understanding**.

Our fictional IT consulting firm needs to evaluate global technology skills and developer demographics. In this notebook, we systematically audit the raw datasets to answer critical foundational questions:
1. **Volume & Granularity**: What is the sample size, dimension, and structure of the dataset?
2. **Schema & Types**: Are column datatypes accurately recognized or will type casting be required?
3. **Data Integrity & Duplicates**: Are there duplicate records that would skew counts and statistical tests?
4. **Missingness Profile**: Which fields suffer from severe missing values, and which core analytical columns require imputation or cautious filtering?
5. **Baseline Distributions**: What do initial descriptive metrics reveal regarding compensation and work hours?

In [ ]:
# Core analytical imports
import os
import json
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup

# Configure Pandas display settings for professional auditing
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

print("Libraries loaded successfully.")

---
### 2. Loading the Raw Developer Survey Dataset
We load the primary raw survey dataset (`data/raw/m1_survey_data.csv`) directly into a Pandas DataFrame.

In [ ]:
SURVEY_PATH = os.path.join("..", "data", "raw", "m1_survey_data.csv")
if not os.path.exists(SURVEY_PATH):
    # Fallback if executing from root directory
    SURVEY_PATH = os.path.join("data", "raw", "m1_survey_data.csv")

df_survey = pd.read_csv(SURVEY_PATH)
num_rows, num_cols = df_survey.shape
print(f"Raw Survey Dataset Successfully Loaded.")
print(f"Total Observations (Rows):    {num_rows:,}")
print(f"Total Attributes (Columns):    {num_cols}")

---
### 3. Data Preview & Structural Inspection
Let us inspect the top 5 records to understand the granularity and preview the format of multi-valued categorical variables.

In [ ]:
df_survey.head(5)

---
### 4. Schema & Data Type Distribution
We audit the data types across all 85 attributes. Notice that several technology fields (e.g., `LanguageWorkedWith`, `DatabaseWorkedWith`, `DevType`) are semicolon-delimited text strings, which will require specialized parsing in the data wrangling phase.

In [ ]:
dtype_counts = df_survey.dtypes.value_counts()
print("Column Data Type Distribution:")
for dt, count in dtype_counts.items():
    print(f"  - {dt}: {count} columns")

print(f"\nTotal memory footprint: {df_survey.memory_usage(deep=True).sum() / (1024 * 1024):.2f} MB")

---
### 5. Duplicate Records Audit
Duplicate submissions introduce severe bias into survey research by double-counting specific respondents.
We inspect duplicates across two levels:
1. **Exact Row Duplicates**: Identical values across all 85 columns.
2. **Primary Key Duplicates**: Duplicate `Respondent` IDs.

In [ ]:
exact_dupes = int(df_survey.duplicated().sum())
id_dupes = int(df_survey.duplicated(subset=['Respondent']).sum())

print(f"Exact Row Duplicates:          {exact_dupes}")
print(f"Duplicate Respondent IDs:      {id_dupes}")

# Preview a sample duplicate respondent to verify matching records
sample_dupe_id = df_survey[df_survey.duplicated(subset=['Respondent'], keep=False)]['Respondent'].iloc[0]
print(f"\nSample duplicate records for Respondent ID {sample_dupe_id}:")
display(df_survey[df_survey['Respondent'] == sample_dupe_id][['Respondent', 'Country', 'DevType', 'LanguageWorkedWith', 'ConvertedComp']])

> **Audit Finding**: Exactly 154 duplicate rows exist in the raw dataset. Deduplication will reduce our sample from **11,552** to **11,398** unique respondents during Phase 5 (Data Cleaning).

---
### 6. Missing Values Analysis
We quantify missingness across the dataset to plan our imputation and filtering strategies.

In [ ]:
missing_counts = df_survey.isna().sum()
missing_df = pd.DataFrame({
    'Missing_Count': missing_counts,
    'Missing_Pct': (missing_counts / len(df_survey)) * 100,
    'Data_Type': df_survey.dtypes
})
missing_df = missing_df[missing_df['Missing_Count'] > 0].sort_values(by='Missing_Count', ascending=False)

print(f"Total columns with missing values: {len(missing_df)} out of {num_cols} ({len(missing_df)/num_cols*100:.1f}%)")
print("\nTop 15 Columns with Highest Missingness:")
display(missing_df.head(15))

---
### 7. Key Analytical Columns Profiling
We inspect the core demographic, compensation, and workplace variables that form the backbone of the consulting project.

In [ ]:
key_attributes = [
    'Country', 'Employment', 'EdLevel', 'DevType', 
    'CompFreq', 'CompTotal', 'ConvertedComp', 
    'LanguageWorkedWith', 'LanguageDesireNextYear', 
    'DatabaseWorkedWith', 'DatabaseDesireNextYear', 
    'WorkLoc', 'Age', 'Gender'
]

profile_rows = []
for col in key_attributes:
    profile_rows.append({
        'Column': col,
        'Data_Type': str(df_survey[col].dtype),
        'Missing_Count': int(df_survey[col].isna().sum()),
        'Missing_Pct': f"{(df_survey[col].isna().sum() / len(df_survey)) * 100:.2f}%",
        'Unique_Values': int(df_survey[col].nunique()),
        'Top_Value': str(df_survey[col].mode().iloc[0]) if not df_survey[col].dropna().empty else 'N/A'
    })

profile_df = pd.DataFrame(profile_rows)
display(profile_df)

---
### 8. Numerical Attributes & Five-Number Summaries
We compute summary statistics for key numerical variables to detect skewness and preliminary anomalies.

In [ ]:
numeric_vars = ['Age', 'WorkWeekHrs', 'CodeRevHrs', 'ConvertedComp']
df_survey[numeric_vars].describe().T[['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']]

#### Preliminary Numerical Observations:
1. **`ConvertedComp`**: Mean ($131,334) is drastically higher than the Median ($57,744) with an extreme maximum of $2,000,000. This indicates severe positive skewness driven by high-earning outliers.
2. **`WorkWeekHrs`**: Maximum value is 1,012 hours/week (an impossibility since a week contains only 168 hours). This represents survey data-entry error that must be filtered or capped.
3. **`Age`**: Median age is 29.0 years (mean 30.77), indicating a relatively young workforce demographic.

---
### 9. Secondary Datasets Inspection
We inspect the secondary job postings dataset (`jobs.json`) and the scraped benchmark table (`Programming_Languages.html`).

In [ ]:
JOBS_PATH = os.path.join("..", "data", "raw", "jobs.json")
if not os.path.exists(JOBS_PATH):
    JOBS_PATH = os.path.join("data", "raw", "jobs.json")

with open(JOBS_PATH, 'r', encoding='utf-8') as f:
    jobs_raw = json.load(f)

df_jobs = pd.DataFrame(jobs_raw)
print(f"Jobs Dataset Dimension: {df_jobs.shape[0]:,} records, {df_jobs.shape[1]} columns.")
print(f"Missing values in Jobs Dataset: {df_jobs.isna().sum().sum()}")
display(df_jobs.head(3))

HTML_PATH = os.path.join("..", "data", "raw", "Programming_Languages.html")
if not os.path.exists(HTML_PATH):
    HTML_PATH = os.path.join("data", "raw", "Programming_Languages.html")

with open(HTML_PATH, 'r', encoding='utf-8') as f:
    soup = BeautifulSoup(f.read(), 'html.parser')

table_rows = soup.find('table').find_all('tr')
header = [th.get_text().strip() for th in table_rows[0].find_all(['th', 'td'])]
rows = [[td.get_text().strip() for td in tr.find_all('td')] for tr in table_rows[1:] if tr.find_all('td')]
df_bench = pd.DataFrame(rows, columns=header)
print(f"\nProgramming Languages Benchmark Table:")
display(df_bench)

---
### 10. Key Takeaways & Recommendations for Subsequent Phases

| Observation | Issue Identified | Action Required in Next Phases |
| :--- | :--- | :--- |
| **Duplicates** | Exactly 154 duplicate rows (and 154 duplicate Respondent IDs). | Remove duplicate rows to retain exactly 11,398 unique respondents in Phase 5. |
| **Missing `WorkLoc`** | 32 missing values in primary working location. | Impute using mode ('Office') during cleaning. |
| **Missing `ConvertedComp`**| 822 missing compensation values (7.12%). | Do not impute arbitrarily; analyze distribution; apply median imputation or document filtration for salary-specific queries. |
| **Anomalous Hours** | `WorkWeekHrs` reaches 1,012 hours. | Treat as data-entry anomalies; apply IQR outlier filtering in Phase 8 (EDA). |
| **Multi-Valued Strings** | Technologies stored as delimited strings. | Semicolon-split and unnest into normalized relational tables in Phase 6 & 7. |

---
### Interview Talking Points: How to Explain Phase 3
> *"Before rushing into model building or visual charts, I conducted a systematic exploratory data assessment. I discovered 154 duplicate respondent IDs, preventing double-counted survey weights. I also discovered severe right-skewness in compensation ($131k mean vs $57k median) and impossible work hours up to 1,012 hrs/week. Profiling these data hygiene issues upfront ensured that our downstream statistical tests and business recommendations were grounded in pristine, validated data."*